In [25]:
import torch
import torch.nn as nn
import torch_pruning as tp
import time

from torchvision.models import mobilenet_v2, resnet18

In [26]:
def repair_quantized_linear_after_pruning(quantized_linear, kept_input_idxs):
    """
    Generic post-hoc repair for a torch.ao.nn.quantized.dynamic.Linear
    whose input dimension no longer matches after upstream structural
    pruning removed some of its input channels.

    Unlike Phase 2's original fix, this does NOT require a separate
    pristine (unquantized) copy of the model. It recovers the trained
    weights directly from the quantized layer itself.
    """
    float_weight = quantized_linear.weight().dequantize()
    float_bias = quantized_linear.bias()

    kept_idxs_tensor = torch.tensor(kept_input_idxs, dtype=torch.long)
    sliced_weight = float_weight[:, kept_idxs_tensor]

    repaired_float_linear = nn.Linear(len(kept_input_idxs), float_weight.shape[0])
    with torch.no_grad():
        repaired_float_linear.weight.copy_(sliced_weight)
        repaired_float_linear.bias.copy_(float_bias)

    repaired_float_linear.qconfig = torch.quantization.default_dynamic_qconfig
    return torch.ao.nn.quantized.dynamic.Linear.from_float(repaired_float_linear)

In [27]:
def get_expected_input_dim(model, classifier_module, input_tensor):
    """
    Detects how many channels are actually arriving at the classifier,
    by hooking its input BEFORE its forward pass runs (avoiding the crash).
    """
    captured = {}
    def hook(module, input):
        captured['dim'] = input[0].shape[1]
    handle = classifier_module.register_forward_pre_hook(hook)
    try:
        with torch.no_grad():
            model(input_tensor)
    except RuntimeError:
        pass
    handle.remove()
    return captured['dim']

In [28]:
example_inputs = torch.randn(1, 3, 224, 224)

quantized_model = torch.quantization.quantize_dynamic(
    mobilenet_v2(weights="DEFAULT").eval(),
    {nn.Linear},
    dtype=torch.qint8
)

pruner = tp.pruner.MagnitudePruner(
    quantized_model, example_inputs,
    importance=tp.importance.MagnitudeImportance(p=2),
    pruning_ratio=0.2,
    ignored_layers=[quantized_model.classifier]
)
pruner.step()

expected_dim = get_expected_input_dim(quantized_model, quantized_model.classifier[1], example_inputs)
declared_dim = quantized_model.classifier[1].in_features
print(f"Classifier expects {declared_dim}, but {expected_dim} channels actually arrive.")

C:\Users\user\AppData\Local\Temp\ipykernel_13876\1850562656.py:3: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  quantized_model = torch.quantization.quantize_dynamic(


Classifier expects 1280, but 1024 channels actually arrive.


In [29]:
history = pruner.pruning_history()
removed_idxs = None
diff = declared_dim - expected_dim

for layer_name, is_out_channel, idxs in history:
    if len(idxs) == diff:
        removed_idxs = idxs
        break

assert removed_idxs is not None, "Could not locate matching pruning event."

all_channels = set(range(declared_dim))
kept_idxs = sorted(list(all_channels - set(removed_idxs)))

repaired_linear = repair_quantized_linear_after_pruning(
    quantized_model.classifier[1], kept_idxs
)
quantized_model.classifier = nn.Sequential(nn.Dropout(0.2), repaired_linear)

with torch.no_grad():
    y = quantized_model(example_inputs)
print("Repaired output shape:", y.shape)

Repaired output shape: torch.Size([1, 1000])


In [30]:
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_dataset = ImageFolder("../data/imagenette2-160/val", transform=transform)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=0)

imagenette_wnid_to_imagenet_idx = {
    "n01440764": 0, "n02102040": 217, "n02979186": 482, "n03000684": 491,
    "n03028079": 497, "n03394916": 566, "n03417042": 569, "n03425413": 571,
    "n03445777": 574, "n03888257": 701,
}
imagenet_indices = [imagenette_wnid_to_imagenet_idx[wnid] for wnid in val_dataset.classes]

def evaluate_accuracy(model, dataloader, imagenet_indices, device="cpu"):
    model.eval()
    model.to(device)
    idx_tensor = torch.tensor(imagenet_indices, device=device)
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            relevant_logits = outputs[:, idx_tensor]
            predicted = torch.argmax(relevant_logits, dim=1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return 100 * correct / total

In [31]:
new_repair_accuracy = evaluate_accuracy(quantized_model, val_loader, imagenet_indices, "cpu")
print(f"MobileNetV2 Q->P (new generalized repair) Accuracy: {new_repair_accuracy:.2f}%")

MobileNetV2 Q->P (new generalized repair) Accuracy: 8.05%


In [32]:
resnet_example_inputs = torch.randn(1, 3, 224, 224)

quantized_resnet = torch.quantization.quantize_dynamic(
    resnet18(weights="DEFAULT").eval(),
    {nn.Linear},
    dtype=torch.qint8
)

resnet_pruner = tp.pruner.MagnitudePruner(
    quantized_resnet, resnet_example_inputs,
    importance=tp.importance.MagnitudeImportance(p=2),
    pruning_ratio=0.2,
    ignored_layers=[quantized_resnet.fc]
)
resnet_pruner.step()

expected_dim_r = get_expected_input_dim(quantized_resnet, quantized_resnet.fc, resnet_example_inputs)
declared_dim_r = quantized_resnet.fc.in_features
print(f"ResNet18 fc expects {declared_dim_r}, but {expected_dim_r} channels actually arrive.")

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\user/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth


100.0%
C:\Users\user\AppData\Local\Temp\ipykernel_13876\1560940903.py:3: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  quantized_resnet = torch.quantization.quantize_dynamic(


ResNet18 fc expects 512, but 409 channels actually arrive.


In [33]:
diff_r = declared_dim_r - expected_dim_r
history_r = resnet_pruner.pruning_history()
removed_idxs_r = None

for layer_name, is_out_channel, idxs in history_r:
    if len(idxs) == diff_r:
        removed_idxs_r = idxs
        break

assert removed_idxs_r is not None, "Could not locate matching pruning event for ResNet18."

all_channels_r = set(range(declared_dim_r))
kept_idxs_r = sorted(list(all_channels_r - set(removed_idxs_r)))

repaired_fc = repair_quantized_linear_after_pruning(quantized_resnet.fc, kept_idxs_r)
quantized_resnet.fc = repaired_fc

with torch.no_grad():
    y_r = quantized_resnet(resnet_example_inputs)
print("Repaired ResNet18 output shape:", y_r.shape)

Repaired ResNet18 output shape: torch.Size([1, 1000])


In [34]:
resnet_repair_accuracy = evaluate_accuracy(quantized_resnet, val_loader, imagenet_indices, "cpu")
print(f"ResNet18 Q->P (generalized repair) Accuracy: {resnet_repair_accuracy:.2f}%")

ResNet18 Q->P (generalized repair) Accuracy: 38.47%


Baseline Accuracy to check if the accuracy above is legit 

In [35]:
baseline_resnet = resnet18(weights="DEFAULT").eval()
resnet_baseline_accuracy = evaluate_accuracy(baseline_resnet, val_loader, imagenet_indices, "cpu")
print(f"ResNet18 Baseline Accuracy: {resnet_baseline_accuracy:.2f}%")

ResNet18 Baseline Accuracy: 97.63%


Importing VGG16 which is the opposite of ResNet-18 in architecture — a long, purely sequential stack of convolutions with no skip connections at all. If our "skip connections protect accuracy" hypothesis is right, VGG16 should collapse hard, similar to MobileNetV2.

In [36]:
from torchvision.models import vgg16

In [37]:
vgg_example_inputs = torch.randn(1, 3, 224, 224)

quantized_vgg = torch.quantization.quantize_dynamic(
    vgg16(weights="DEFAULT").eval(),
    {nn.Linear},
    dtype=torch.qint8
)

vgg_pruner = tp.pruner.MagnitudePruner(
    quantized_vgg, vgg_example_inputs,
    importance=tp.importance.MagnitudeImportance(p=2),
    pruning_ratio=0.2,
    ignored_layers=[quantized_vgg.classifier]
)
vgg_pruner.step()

vgg_first_linear = quantized_vgg.classifier[0]
expected_dim_v = get_expected_input_dim(quantized_vgg, vgg_first_linear, vgg_example_inputs)
declared_dim_v = vgg_first_linear.in_features
print(f"VGG16 classifier[0] expects {declared_dim_v}, but {expected_dim_v} channels actually arrive.")

Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to C:\Users\user/.cache\torch\hub\checkpoints\vgg16-397923af.pth


100.0%
C:\Users\user\AppData\Local\Temp\ipykernel_13876\374311815.py:3: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  quantized_vgg = torch.quantization.quantize_dynamic(


VGG16 classifier[0] expects 25088, but 20041 channels actually arrive.


In [40]:
last_conv_name = None
for name, module in quantized_vgg.named_modules():
    if isinstance(module, nn.Conv2d):
        last_conv_name = name

print("Last conv layer:", last_conv_name)

removed_idxs_v = None
for layer_name, is_out_channel, idxs in history_v:
    if layer_name == last_conv_name and is_out_channel:
        removed_idxs_v = idxs
        break

assert removed_idxs_v is not None, f"Could not find pruning event for {last_conv_name}"

original_channels = 512
spatial_size = 7 * 7

all_channels_v = set(range(original_channels))
kept_channels_v = sorted(list(all_channels_v - set(removed_idxs_v)))

kept_idxs_v = []
for ch in kept_channels_v:
    start = ch * spatial_size
    kept_idxs_v.extend(range(start, start + spatial_size))

print(f"Kept channels: {len(kept_channels_v)}, expanded flattened slots: {len(kept_idxs_v)}")
assert len(kept_idxs_v) == expected_dim_v, "Expanded slot count doesn't match observed input dimension."

repaired_vgg_linear = repair_quantized_linear_after_pruning(vgg_first_linear, kept_idxs_v)
quantized_vgg.classifier[0] = repaired_vgg_linear

with torch.no_grad():
    y_v = quantized_vgg(vgg_example_inputs)
print("Repaired VGG16 output shape:", y_v.shape)

Last conv layer: features.28
Kept channels: 409, expanded flattened slots: 20041
Repaired VGG16 output shape: torch.Size([1, 1000])


In [41]:
vgg_repair_accuracy = evaluate_accuracy(quantized_vgg, val_loader, imagenet_indices, "cpu")
print(f"VGG16 Q->P (generalized repair) Accuracy: {vgg_repair_accuracy:.2f}%")

baseline_vgg = vgg16(weights="DEFAULT").eval()
vgg_baseline_accuracy = evaluate_accuracy(baseline_vgg, val_loader, imagenet_indices, "cpu")
print(f"VGG16 Baseline Accuracy: {vgg_baseline_accuracy:.2f}%")

VGG16 Q->P (generalized repair) Accuracy: 33.35%
VGG16 Baseline Accuracy: 97.30%


Does EfficientNet-B0 — a different network, but built from the same core building block as MobileNetV2 (depthwise-separable convolutions in MBConv blocks) — also collapse hard under the same 20% pruning, similar to MobileNetV2's ~89-point drop? Or does it behave more like ResNet-18 and VGG16 (a ~55-65 point drop)?

In [42]:
from torchvision.models import efficientnet_b0

In [43]:
efficientnet_example_inputs = torch.randn(1, 3, 224, 224)

quantized_efficientnet = torch.quantization.quantize_dynamic(
    efficientnet_b0(weights="DEFAULT").eval(),
    {nn.Linear},
    dtype=torch.qint8
)

efficientnet_pruner = tp.pruner.MagnitudePruner(
    quantized_efficientnet, efficientnet_example_inputs,
    importance=tp.importance.MagnitudeImportance(p=2),
    pruning_ratio=0.2,
    ignored_layers=[quantized_efficientnet.classifier]
)
efficientnet_pruner.step()

efficientnet_linear = quantized_efficientnet.classifier[1]
expected_dim_e = get_expected_input_dim(quantized_efficientnet, efficientnet_linear, efficientnet_example_inputs)
declared_dim_e = efficientnet_linear.in_features
print(f"EfficientNet-B0 classifier[1] expects {declared_dim_e}, but {expected_dim_e} channels actually arrive.")

C:\Users\user\AppData\Local\Temp\ipykernel_13876\2516650561.py:3: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  quantized_efficientnet = torch.quantization.quantize_dynamic(


EfficientNet-B0 classifier[1] expects 1280, but 1024 channels actually arrive.


In [44]:
last_conv_name_e = None
for name, module in quantized_efficientnet.named_modules():
    if isinstance(module, nn.Conv2d):
        last_conv_name_e = name

print("Last conv layer:", last_conv_name_e)

history_e = efficientnet_pruner.pruning_history()
removed_idxs_e = None
for layer_name, is_out_channel, idxs in history_e:
    if layer_name == last_conv_name_e and is_out_channel:
        removed_idxs_e = idxs
        break

assert removed_idxs_e is not None, f"Could not find pruning event for {last_conv_name_e}"

all_channels_e = set(range(declared_dim_e))
kept_idxs_e = sorted(list(all_channels_e - set(removed_idxs_e)))

repaired_efficientnet_linear = repair_quantized_linear_after_pruning(efficientnet_linear, kept_idxs_e)
quantized_efficientnet.classifier[1] = repaired_efficientnet_linear

with torch.no_grad():
    y_e = quantized_efficientnet(efficientnet_example_inputs)
print("Repaired EfficientNet-B0 output shape:", y_e.shape)

Last conv layer: features.8.0
Repaired EfficientNet-B0 output shape: torch.Size([1, 1000])


In [45]:
efficientnet_repair_accuracy = evaluate_accuracy(quantized_efficientnet, val_loader, imagenet_indices, "cpu")
print(f"EfficientNet-B0 Q->P (generalized repair) Accuracy: {efficientnet_repair_accuracy:.2f}%")

baseline_efficientnet = efficientnet_b0(weights="DEFAULT").eval()
efficientnet_baseline_accuracy = evaluate_accuracy(baseline_efficientnet, val_loader, imagenet_indices, "cpu")
print(f"EfficientNet-B0 Baseline Accuracy: {efficientnet_baseline_accuracy:.2f}%")

EfficientNet-B0 Q->P (generalized repair) Accuracy: 8.97%
EfficientNet-B0 Baseline Accuracy: 99.41%
